In [6]:
import pandas as pd
import numpy as np
from pathlib import Path

# Calculation of P(t) for each hour of the week

In [7]:
# Load price data and set datetime index
df_prod = pd.read_excel(
    "../data/Input backtest formule Jimmy/Production_Plan_Touluch_01.10.2025-30.04.2026.xlsx",
    parse_dates=["time"]
).set_index("time")

df_prod.index = df_prod.index.tz_localize("Europe/Paris", ambiguous="infer").tz_convert("UTC")

# Load price data and set datetime index
df_price = pd.read_excel(
    "../data/Input backtest formule Jimmy/DA prices_01.10.2025-30.04.2026.xlsx",
    parse_dates=["datetime"]
).set_index("datetime")

df_price.index = df_price.index.tz_localize("Europe/Paris", ambiguous="infer").tz_convert("UTC")

# Join datasets on datetime index
df = df_prod.join(df_price, how="inner").sort_index()

# Je récupère les 5 jours glissants
df["date"] = df.index.floor("D")

# Calcul jour par jour
results = []

unique_days = df["date"].drop_duplicates().sort_values()

for day in unique_days:

    # fenêtre : jour D jusqu'à D+5
    end_day = day + pd.Timedelta(days=5)

    window = df[(df.index >= day) & (df.index < end_day)]

    # garder seulement moments avec production
    window_prod = window[window["value"] > 0]

    if len(window_prod) > 0:
        p_min = window_prod["price"].min()
    else:
        p_min = None  # ou np.nan

    results.append({
        "date": day,
        "P_t": p_min
    })

# fallback
df_daily = pd.DataFrame(results).set_index("date")

df = df.join(df_daily, on="date")

df["P_t"] = df["P_t"].ffill()

result = df[["P_t"]]

print(result.head(20))

                             P_t
time                            
2025-09-30 22:00:00+00:00  87.56
2025-09-30 23:00:00+00:00  87.56
2025-10-01 00:00:00+00:00  87.56
2025-10-01 01:00:00+00:00  87.56
2025-10-01 02:00:00+00:00  87.56
2025-10-01 03:00:00+00:00  87.56
2025-10-01 04:00:00+00:00  87.56
2025-10-01 05:00:00+00:00  87.56
2025-10-01 06:00:00+00:00  87.56
2025-10-01 07:00:00+00:00  87.56
2025-10-01 08:00:00+00:00  87.56
2025-10-01 09:00:00+00:00  87.56
2025-10-01 10:00:00+00:00  87.56
2025-10-01 11:00:00+00:00  87.56
2025-10-01 12:00:00+00:00  87.56
2025-10-01 13:00:00+00:00  87.56
2025-10-01 14:00:00+00:00  87.56
2025-10-01 15:00:00+00:00  87.56
2025-10-01 16:00:00+00:00  87.56
2025-10-01 17:00:00+00:00  87.56


#   Calculation of V(t) for each hour of the week

In [8]:
# Load Galens reservoir data
df_res = pd.read_csv(
    "../data/Input backtest formule Jimmy/RL(m3)_Galens_R_01.10.25-30.04.26.csv",
    parse_dates=["datetime_utc"]
).set_index("datetime_utc")

df_res.index = df_res.index.tz_localize("UTC")

# Conversion volume to power
def volume_to_power(volume):
    volume = volume/1000 # m3 = Mm3
    if volume < 733:
        return 9.0
    else:
        return 9.5

df_res["P_reservoir"] = df_res["value"].apply(volume_to_power)

# Calcul Pmax(t)
Pmax_theorique = 9.5
indispo = 0

df_res["P_max"] = np.minimum(
    Pmax_theorique * (1 - indispo),
    df_res["P_reservoir"]
)

# Result
df_res_hourly = (df_res["P_max"].groupby(df_res.index.floor("h")).min().to_frame(name="P_max"))

print(df_res_hourly.head(10))
print(df_res_hourly.shape)

                           P_max
datetime_utc                    
2025-09-30 22:00:00+00:00    9.0
2025-09-30 23:00:00+00:00    9.0
2025-10-01 00:00:00+00:00    9.0
2025-10-01 01:00:00+00:00    9.0
2025-10-01 02:00:00+00:00    9.0
2025-10-01 03:00:00+00:00    9.0
2025-10-01 04:00:00+00:00    9.0
2025-10-01 05:00:00+00:00    9.0
2025-10-01 06:00:00+00:00    9.0
2025-10-01 07:00:00+00:00    9.0
(5065, 1)


In [9]:
# Check index alignment before joining
idx_res = df_res_hourly.index.sort_values()
idx_result = result.index.sort_values()

only_in_res = idx_res.difference(idx_result)
only_in_result = idx_result.difference(idx_res)

print(f"In df_res_hourly but not in result : {len(only_in_res)}")
print(f"In result but not in df_res_hourly : {len(only_in_result)}")

if len(only_in_res) == 0 and len(only_in_result) == 0:
    print("✅ Indexes perfectly aligned — join without data loss")
else:
    print("⚠️ Indexes not aligned — join will be partial")
    print("Missing in result      :", only_in_res[:5])
    print("Missing in df_res_hourly:", only_in_result[:5])

# Join on datetime index
df_final = result.join(df_res_hourly, how="inner")

print(f"\nShape result        : {result.shape}")
print(f"Shape df_res_hourly : {df_res_hourly.shape}")
print(f"Shape df_final      : {df_final.shape}")
df_final.index.name = 'datetime'
print(df_final.head(10))

In df_res_hourly but not in result : 0
In result but not in df_res_hourly : 0
✅ Indexes perfectly aligned — join without data loss

Shape result        : (5065, 1)
Shape df_res_hourly : (5065, 1)
Shape df_final      : (5065, 2)
                             P_t  P_max
datetime                               
2025-09-30 22:00:00+00:00  87.56    9.0
2025-09-30 23:00:00+00:00  87.56    9.0
2025-10-01 00:00:00+00:00  87.56    9.0
2025-10-01 01:00:00+00:00  87.56    9.0
2025-10-01 02:00:00+00:00  87.56    9.0
2025-10-01 03:00:00+00:00  87.56    9.0
2025-10-01 04:00:00+00:00  87.56    9.0
2025-10-01 05:00:00+00:00  87.56    9.0
2025-10-01 06:00:00+00:00  87.56    9.0
2025-10-01 07:00:00+00:00  87.56    9.0


In [ ]:
if not Path("../data/P_n_V.csv").exists():
    df_final.to_csv("../data/P_n_V.csv")
    print('P_n_V.csv created successfully.')
else:
    print('P_n_V.csv already exists. No new file created.')